# 🐍 Master Guide: 04: Encapsulation, Name Mangling & `@property` Getters/Setters

> **A Complete Foundations-to-Advanced Reference Guide**  
> Covers: Encapsulation Philosophy, Public vs. Protected vs. Private Access Conventions, Name Mangling (`_Class__attr`), Traditional Getters/Setters, and Pythonic `@property` Validation.
---

## 1. Core Definitions: Encapsulation & Abstraction

### What is Data Encapsulation?
> **Definition**: **Data Encapsulation** is the process of binding data members (attributes) and the methods that operate on that data into a single cohesive unit (Class), while **restricting direct external access** to critical internal variables.

### The 3 Access Modifiers in Python:

| Access Modifier | Naming Pattern | Visibility / Intended Scope | Enforcement Mechanism |
|---|---|---|---|
| **Public** | `name` | Accessible from anywhere (inside and outside class). | No restriction. |
| **Protected** | `_name` | Intended for use inside the class and its subclasses only. | **Gentleman's Agreement / Convention** (PEP 8). Still technically accessible. |
| **Private** | `__name` | Strictly internal to the defining class. | **Enforced via Name Mangling**: Renamed to `_ClassName__name`. |

## 2. Name Mangling Architecture Drill

When an attribute begins with double underscores (and at most one trailing underscore), the Python compiler automatically alters the identifier by prefixing `_ClassName`:

$$\text{self.\_\_pin} \quad \xrightarrow{\text{CPython Compiler}} \quad \text{self.\_BankAccount\_\_pin}$$

This prevents accidental overriding by derived classes and signals strict private encapsulation.

In [ ]:
# Encapsulation & Name Mangling Demonstration
class SecureBankAccount:
    def __init__(self, acc_no, holder, pin, initial_balance):
        self.acc_no = acc_no          # Public attribute
        self._holder = holder         # Protected attribute (Convention)
        self.__pin = pin              # Private attribute (Name mangled)
        self.__balance = initial_balance  # Private attribute (Name mangled)

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive.")
        self.__balance += amount
        print(f"Deposited Rs. {amount:,.2f}. New Balance: Rs. {self.__balance:,.2f}")

    def withdraw(self, amount, entered_pin):
        if entered_pin != self.__pin:
            print("SECURITY ALERT: Incorrect PIN entered! Transaction denied.")
            return False
        if amount > self.__balance:
            print("TRANSACTION REJECTED: Insufficient funds available.")
            return False
        self.__balance -= amount
        print(f"Dispensed Rs. {amount:,.2f}. Remaining Balance: Rs. {self.__balance:,.2f}")
        return True

acc = SecureBankAccount("AC-987654", "Aravind Rao", 4321, 50000.0)
acc.deposit(10000.0)
acc.withdraw(15000.0, entered_pin=4321)  # Success
acc.withdraw(5000.0, entered_pin=9999)   # Wrong PIN

# Direct access to __balance raises AttributeError:
try:
    print(acc.__balance)
except AttributeError as err:
    print(f"\nDirect private access failed as expected: {err}")

# Inspecting Name Mangling in __dict__
print("Name Mangled attributes in object dictionary:")
for key in acc.__dict__:
    print(f"  {key}")

## 3. Pythonic `@property` Getters & Setters

Traditional languages require ugly method calls like `obj.getBalance()` and `obj.setBalance(val)`.
Python provides the **`@property` decorator**, allowing methods to be accessed transparently using **natural attribute syntax** (`obj.balance`) while secretly executing validation logic behind the scenes!

In [ ]:
# Encapsulation with @property, @setter, and @deleter
class TemperatureSensor:
    def __init__(self, celsius=20.0):
        self._celsius = celsius  # Protected backing variable

    # 1. Getter property
    @property
    def celsius(self):
        return self._celsius

    # 2. Setter with validation
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Physically impossible: Temperature cannot be below Absolute Zero (-273.15 C)!")
        self._celsius = value

    # 3. Computed dynamic property (read-only)
    @property
    def fahrenheit(self):
        return (self._celsius * 9/5) + 32.0

sensor = TemperatureSensor(25.0)
print(f"Current Reading : {sensor.celsius:.1f} C | {sensor.fahrenheit:.1f} F")

# Modifying via clean attribute syntax
sensor.celsius = 37.0
print(f"Body Temp       : {sensor.celsius:.1f} C | {sensor.fahrenheit:.1f} F")

# Triggering validation exception
try:
    sensor.celsius = -300.0
except ValueError as err:
    print("Validation Guard Caught:", err)

## 4. Summary & Quick Revision Cheat Sheet

| Access Level | Syntax | CPython Compiler Handling | Best Practice Role |
|---|---|---|---|
| **Public** | `self.name` | Unaltered | Standard open API. |
| **Protected** | `self._name` | Unaltered (Convention only) | Internal to hierarchy. |
| **Private** | `self.__name` | Mangled to `_ClassName__name` | Sensitive state; prevent collision. |
| **Getter Property** | `@property def x(self):` | Wrapped in descriptor | Read access via `obj.x`. |
| **Setter Property** | `@x.setter def x(self, val):`| Validates on assignment | Write access via `obj.x = val`. |